In [1]:
# Setup: Import modules and define paths
from pathlib import Path
import pandas as pd
import sys

# Find repo root
repo_root = Path.cwd()
for candidate in [repo_root] + list(repo_root.parents):
    if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
        repo_root = candidate
        break

# Add repo root to sys.path FIRST (before importing from functions)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Now import validation functions
from functions.validation_functions import (
    load_groundwater_csv,
    #FIXME: Hier misschien nog aan toevoegen om te beginn bij eerste valid timestamp?
    remove_duplicate_and_fill_missing,
    #FIXME: Hier moet nog een functie bij voor eenheden checken (boven 100cm)
    flag_physical_bounds,
    flag_unrealistic_step_change,
    flag_constant_head_periods,
    flag_statistical_outliers,
)

# Define folders
wiertsema_input_dir = repo_root / 'output_data' / 'csv_wiertsema_knmi'
fugro_input_dir = repo_root / 'output_data' / 'csv_fugro_knmi'

print('Setup complete!')
print(f'Repo root: {repo_root}')
print(f'Wiertsema input: {wiertsema_input_dir}')
print(f'Fugro input: {fugro_input_dir}')

Setup complete!
Repo root: d:\Users\jvanruitenbeek\data_validation
Wiertsema input: d:\Users\jvanruitenbeek\data_validation\output_data\csv_wiertsema_knmi
Fugro input: d:\Users\jvanruitenbeek\data_validation\output_data\csv_fugro_knmi


# Batch Processing (All Files)


In [2]:
# Step 1: Define validation parameters (adjust these based on your data)

# Physical bounds (meters NAP or similar datum)
#hmin = -10  # Minimum realistic head value
#hmax = 10   # Maximum realistic head value #
# CURRENTLY DEFINED IN LOOP

# Maximum allowed rate of change between timestamps in m/day
max_up = 0.3
max_down = -0.05  

# Constant head periods
tconst_steps = 24   # Flag if at least 24 hourly measurements are constant
flat_margin_m = 0.02  # Checking the 'dH' column for changes smaller than this
min_band_m = 0.15   # Using the minimum value from the head series, then adding this margin on top


# print('Validation parameters set:')
# print(f'  Physical bounds: {hmin} to {hmax} m')
#print(f'  Constant head: >{tconst_days} days or >{nconst_min} measurements')

In [3]:
# Configuration: Choose which folder to process
# Options: 'wiertsema' or 'fugro'
folder_choice = 'wiertsema'

# Select input and output folders based on choice
if folder_choice.lower() == 'wiertsema':
    input_folder = wiertsema_input_dir
    output_folder = repo_root / 'output_data' / 'csv_wiertsema_validated'
    print('Selected: WIERTSEMA')
elif folder_choice.lower() == 'fugro':
    input_folder = fugro_input_dir
    output_folder = repo_root / 'output_data' / 'csv_fugro_validated'
    print('Selected: FUGRO')
else:
    raise ValueError("folder_choice must be 'wiertsema' or 'fugro'")

# Create output folder if it doesn't exist
output_folder.mkdir(parents=True, exist_ok=True)

print("   Batch Processing Configuration:")
print(f"  Input folder:  {input_folder}")
print(f"  Output folder: {output_folder}")
#print(f"  Parameters: hmin={hmin}, hmax={hmax}, max_up={max_up}, max_down={max_down}")
print(f"  Other parameters: tconst_steps={tconst_steps}, flat_margin_m={flat_margin_m}, min_band_m={min_band_m}")

Selected: WIERTSEMA
   Batch Processing Configuration:
  Input folder:  d:\Users\jvanruitenbeek\data_validation\output_data\csv_wiertsema_knmi
  Output folder: d:\Users\jvanruitenbeek\data_validation\output_data\csv_wiertsema_validated
  Other parameters: tconst_steps=24, flat_margin_m=0.02, min_band_m=0.15


In [4]:
# Loading object data
object_data = pd.read_csv(repo_root / 'output_data' / 'object_data.csv', index_col=0)

In [5]:
# Load object data ONCE before the loop (better & faster)
object_data = pd.read_csv(repo_root / "output_data" / "object_data.csv", index_col=0)

print("object_data index sample (first 5 rows):")
for idx in list(object_data.index[:5]):
    print("  ", repr(idx))
print()

# Execute batch processing
csv_files = sorted(input_folder.glob('*.csv'))
print(f"Found {len(csv_files)} CSV files to process\n")

results = []
failed_files = []

for i, csv_file in enumerate(csv_files, 1):
    print(i,csv_file)
    try:
        print(f"[{i}/{len(csv_files)}] Processing: {csv_file.name}")

        # Load sensor data
        df = load_groundwater_csv(csv_file)
        initial_rows = len(df)
        print(f"  Initial rows: {initial_rows}")

        # -------------------------------------------------------
        # OPTIONAL PARAMETER OVERRIDE FROM object_data.csv
        # -------------------------------------------------------
        # Defaults
        hmin = -10
        hmax = 10

        fname = csv_file.name
        print(f"  Looking for match in object_data for: {repr(fname)}")

        if fname in object_data.index:
            print("  ✓ Match found in object_data.csv")
            row = object_data.loc[fname]

            # Override hmin if available
            if "hmin_pb" in row.index and not pd.isna(row["hmin_pb"]):
                hmin = float(row["hmin_pb"])
                print(f"    ➜ Using object-defined hmin = {hmin}")
            else:
                print("    (No hmin_pb defined, keeping default hmin)")

            # Override hmax if available
            if "h_hmax_pb" in row.index and not pd.isna(row["h_hmax_pb"]):
                hmax = float(row["h_hmax_pb"])
                print(f"    ➜ Using object-defined hmax = {hmax}")
            else:
                print("    (No h_hmax_pb defined, keeping default hmax)")
        else:
            print("  ⚠ No match found in object_data.csv, using defaults.")

        # Final parameters used for this file
        print(f"  Final hmin = {hmin}, hmax = {hmax}")

            # Load the CSV
        df = load_groundwater_csv(csv_file)
        initial_rows = len(df)

        # Printing info the command
        print(df.info())

        # Print rows
        print(f"  Initial rows: {initial_rows}")
        
        # Step 1: Remove duplicates
        #df_1, _ = remove_duplicate_and_fill_missing(df)
        
        # Step 2: Flag physical bounds
        df_2, _ = flag_physical_bounds(df, hmin, hmax)
        
        # Step 3: Flag unrealistic steps
        df_3, _ = flag_unrealistic_step_change(df_2, max_up, max_down)
        
        # Step 4: Flag constant head periods
        df_4, _ = flag_constant_head_periods(df_3, tconst_steps, flat_margin_m, min_band_m, hmin)
        
        # Step 5: Flag statistical outliers
        df_5, _ = flag_statistical_outliers(df_4)
        
        # Clean up temporary column
        df_final = df_5.drop(columns=["dH"], errors='ignore')
        
        # Save to output
        output_file = output_folder / f'{csv_file.stem}.csv'
        df_final.to_csv(output_file, index=False)
        
        # Count flagged rows
        flag_cols = [col for col in df_final.columns if col.startswith('v')]
        flagged_count = df_final[flag_cols].notna().any(axis=1).sum()
        
        results.append({
            'Filename': csv_file.name,
            'Status': '✓',
            'Initial Rows': initial_rows,
            'Final Rows': len(df_final),
            'Flagged Points': flagged_count,
        })
        print(f"  ✓ Saved to {output_file.name}")
        print(f"    Initial: {initial_rows} rows → Final: {len(df_final)} rows, {flagged_count} flagged\n")
        
    except Exception as e:
        print(f"  ✗ Error: {str(e)}\n")
        failed_files.append(csv_file.name)
        results.append({
            'Filename': csv_file.name,
            'Status': '✗',
            'Initial Rows': 0,
            'Final Rows': 0,
            'Flagged Points': 0,
        })

# Summary
print("="*70)
print("BATCH PROCESSING COMPLETE")
print("="*70)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f"\nTotal files processed: {len(results_df)}")
print(f"Successful: {len(results_df[results_df['Status'] == '✓'])}")
print(f"Failed: {len(failed_files)}")
if failed_files:
    print("\nFailed files:")
    for fname in failed_files:
        print(f"  - {fname}")

object_data index sample (first 5 rows):
   'NL-2412417-HWM_B09-PB1_m_NAP_avg.csv'
   'NL-2412417-HWM_B09-PB2_m_NAP_avg.csv'
   'NL-2412417-HWM_B12-PB1_m_NAP_avg.csv'
   'NL-2412417-HWM_B13-PB1_m_NAP_avg.csv'
   'NL-2412417-HWM_B13-PB2_m_NAP_avg.csv'

Found 302 CSV files to process

1 d:\Users\jvanruitenbeek\data_validation\output_data\csv_wiertsema_knmi\83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv
[1/302] Processing: 83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv
 Length of df before cleaning: 3795
ℹ️  33 NaN head values in 'd:\Users\jvanruitenbeek\data_validation\output_data\csv_wiertsema_knmi\83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv' (kept, not dropped)
  Initial rows: 3795
  Looking for match in object_data for: '83034-1 HB001PB01 BE0049+00_BUKR_GMW_PB1_F-229.csv'
  ✓ Match found in object_data.csv
    ➜ Using object-defined hmin = -2.79
    (No h_hmax_pb defined, keeping default hmax)
  Final hmin = -2.79, hmax = 10
 Length of df before cleaning: 3795
ℹ️  33 Na